# Week 4: Exploratory Data Analysis & Signal Audit (ML-06)

**Track:** Applied Search Intelligence: Google Search Ranking & Discoverability  
**Lane:** Refresh / Content Opportunity Scoring  
**Author:** Altaf (ML Intern)  
**Dataset:** Starter Anonymized Release (`data/raw/content_refresh_anonymized.csv`)  

---

## 1. Distribution Diagnostics & Heavy-Tail Handling

Search engine telemetry follows extreme power-law (heavy-tailed) distributions:
- The top 5% of URLs generate over 68% of aggregate impressions and clicks.
- Raw Pearson correlation coefficients are heavily distorted by extreme outliers.
- We apply log1p transforms ($\log(1+x)$) to all volume telemetry (`impressions_90d`, `clicks_90d`, `sessions_90d`) and evaluate relationships via grouped tier medians.

## 2. Verdict-Style Signal Mini-Tests

Following `skills/auditing-signals/SKILL.md`, each hypothesis is tested with sample-size floors ($n \ge 50$) and assigned an explicit verdict: **CONFIRMED**, **OPPOSITE**, **MIXED**, or **FALSE**.

### Test 1: "Content staleness ($\ge 180$ days since update) accelerates click decline risk."
- **Test Design:** Compare empirical decline rates (`is_declining_label`) across freshness tiers.
- **Observed Data:**
  - Stale ($>180$ days since update): $n = 14,210$, Decline Rate = **61.4%**
  - Moderately Fresh (91–180 days): $n = 8,940$, Decline Rate = **51.8%**
  - Recently Updated ($\le 90$ days): $n = 6,850$, Decline Rate = **44.8%**
- **Verdict:** **CONFIRMED**
- **Operational Meaning:** Staleness is a powerful risk factor (+16.6% absolute risk increase), justifying inclusion in both the heuristic baseline and learned models.

---

### Test 2: "Longer content depth (>2,000 words) protects against search decline."
- **Test Design:** Compare decline rates across word count tiers.
- **Observed Data:**
  - Short (<1,000 words): $n = 7,420$, Decline Rate = **55.1%**
  - Medium (1,000–2,000 words): $n = 16,110$, Decline Rate = **53.9%**
  - Long (>2,000 words): $n = 6,470$, Decline Rate = **54.0%**
- **Verdict:** **FALSE / NO EFFECT**
- **Operational Meaning:** Contrary to popular SEO folklore, text length alone provides no structural barrier against decay once content becomes stale or intent shifts. Word count carries low feature importance (0.0397).

---

### Test 3: "Low CTR (<0.5%) on visible Page 1 pages signals SERP snippet intent mismatch."
- **Test Design:** Evaluate decline rates on high-visibility pages (impressions $\ge 500$, pos $\le 10$) with CTR $< 0.5\%$.
- **Observed Data:**
  - Visible with Low CTR (<0.5%): $n = 4,120$, Decline Rate = **64.8%**
  - Visible with Healthy CTR ($\ge 0.5\%$): $n = 12,850$, Decline Rate = **48.2%**
- **Verdict:** **CONFIRMED**
- **Operational Meaning:** High impressions with depressed CTR indicate that searchers bypass the snippet. This justifies the dedicated operational action: `refresh_and_review_ctr`.

---

### Test 4: "Low reader engagement (<30% scroll rate) predicts organic decline."
- **Test Design:** Filter sessions $\ge 30$, compare scroll rate $< 30\%$ vs $\ge 30\%$.
- **Observed Data:**
  - Low Scroll (<30%): $n = 3,890$, Decline Rate = **62.3%**
  - High Scroll ($\ge 30\%$): $n = 8,140$, Decline Rate = **49.1%**
- **Verdict:** **CONFIRMED**
- **Operational Meaning:** Reader dissatisfaction directly precedes Google ranking drops. Justifies the dedicated action: `refresh_and_review_engagement`.

## 3. Signal Audit Summary

| Candidate Signal | Hypothesis Tested | Sample Size ($n$) | Observed Contrast | Verdict |
|---|---|---:|---|---|
| `days_since_last_update` | Staleness increases decay | 30,000 | 61.4% vs 44.8% | **CONFIRMED** |
| `word_count` | Length prevents decay | 30,000 | 54.0% vs 55.1% | **FALSE** |
| `ctr` on Page 1 | Low CTR leads to drop | 16,970 | 64.8% vs 48.2% | **CONFIRMED** |
| `scroll_rate` | Low engagement leads to drop | 12,030 | 62.3% vs 49.1% | **CONFIRMED** |